# MARV × Titans — does the forgetting curve hold with REAL text? (Colab T4)

Every experiment on `marv-titan` so far (`titans_memdiff.py`, `titans_ablation.py`,
`titans_per_unit.py`) fed the memory random 64-dim vectors. That's deliberate — it isolates
the memory mechanism from language modeling — but it leaves open whether the same forgetting
curve and diffuse-storage story hold once the memory is reading actual language instead of
noise, and whether real language creates any structure random vectors can't.

This notebook trains a small, real, byte-level language model with a Titans memory wired in
(`titans_pytorch.MemoryAsContextTransformer`, the "MAC" architecture) on enwik8 (Wikipedia
text), then re-runs the same early-vs-end weight-snapshot diff on the memory as it reads a real
held-out passage.

**Scaled down from the library's own `train_mac.py` recipe on purpose** (that one is dim=384,
depth=8, 100k batches, wandb, flex-attention — a real multi-hour+ run, not a Colab demo). Here:
dim=64 (matches the memory's own `dim_head`, so results are directly comparable to the
random-vector notebooks), depth=4, one memory layer, no flex-attention. This will NOT produce
a good language model — it's a correctness + qualitative-structure check, not a real LM.

In [ ]:
!pip install -q titans-pytorch
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-pytorch-src
import sys; sys.path.insert(0, '/content/marv/experiments')

import os
# the enwik8 dataset ships inside the titans-pytorch repo itself (data/enwik8.gz) --
# if this ever moves, download it directly from http://prize.hutter1.net/ instead.
DATA_PATH = '/content/titans-pytorch-src/data/enwik8.gz'
assert os.path.exists(DATA_PATH), 'enwik8.gz not found -- see the comment above for a fallback source'

import time, numpy as np, torch, matplotlib.pyplot as plt
from titans_real_text import build_model, load_enwik8, sample_batch, train, diff_memory_on_passage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('device:', device)

## 1. Load enwik8, build the small MAC transformer

`neural_memory_model=MemoryMLP(64, depth=2, expansion_factor=4.)` gives the memory the exact
same 64→256→64 shape used in every prior notebook on this branch — so the numbers below are
directly comparable to the random-vector experiments, not just qualitatively similar.

In [ ]:
data_train, data_val = load_enwik8(DATA_PATH)
print(f'train bytes: {len(data_train):,}   val bytes: {len(data_val):,}')

model = build_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model: {n_params/1e6:.2f}M params')

## 2. Train

On a T4 this should run noticeably faster than the ~1.3s/step seen on CPU locally — feel free
to raise `STEPS` well past the default if you want a stronger (if still small) model. The loss
will not get near the library's own reported numbers at this scale; watch for it dropping
below the byte-uniform baseline (`ln(256) ≈ 5.545` nats) and continuing to fall, which is the
actual thing being checked here.

In [ ]:
STEPS = 3000
SEQ_LEN = 256
BATCH_SIZE = 16

t0 = time.time()
train(model, data_train, data_val, steps=STEPS, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, lr=2e-4, device=device)
print(f'\n{STEPS} steps took {time.time()-t0:.0f}s')

## 3. Diff the memory on a real held-out passage

Same early-vs-end snapshot diff as `titans_memdiff.py` (`gate_cos`, `norm_ratio`, direction/
magnitude retained on the hardest-written units) — except the "document" is now real English
text from held-out enwik8, not 96 random vectors.

In [ ]:
passage = sample_batch(data_val, 512, 1)[0]
print('passage (decoded):')
print(repr(bytes(passage[:200].tolist()).decode('utf-8', errors='replace')))
print()
diff_memory_on_passage(model, passage, device)

## What to look for

- **Does `norm_ratio` on the moved units still show a clear pattern** (writes decaying or
  accumulating), the way the random-vector version did? Real text is far from IID random
  vectors — repeated bytes, common words, and predictable structure could make the memory's
  write/forget behavior look different (e.g. less each token is genuinely "surprising").
- **Does the direction-retained / magnitude-retained pattern for the hardest-written units**
  look like the same exponential-ish forgetting curve, or does real text's redundancy change
  the shape?
- This is still a small, briefly-trained model — a difference here could mean "real text
  changes the story" or just "this model hasn't trained long enough to show it." Worth
  comparing at a few different `STEPS` values before drawing a firm conclusion.

**Next steps (not done here):** a `describe_feature`-style logit lens reading what a specific
hidden unit promotes (e.g. "unit 33 now writes toward the letter e"), and re-running the
`titans_per_unit.py` per-unit-decay localization test with this real-text-trained memory
instead of random vectors. See `experiments/README.md` roadmap item 3 on the `marv-titan`
branch.